# Machine Learning Lab: Classification and Regression

## Introduction

The purpose of this lab is to apply machine learning techniques to classification and regression problems. The lab focuses on building reliable machine learning pipelines, comparing different models using cross-validation, tuning model hyperparameters, and evaluating the selected models on unseen test data.

Two experiments are conducted:

- **Experiment A: Classification** – Predict whether a support ticket will be escalated or not.
- **Experiment B: Regression** – Predict the resolution time of a support ticket in hours.

The experiments use preprocessing pipelines to handle missing values and prepare the data for machine learning models. Cross-validation is used to compare model performance, while a separate test set is reserved for final evaluation.

The main objective is to select suitable models while preventing data leakage and understanding the limitations of the results.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)
clf_df = pd.read_csv("classification_escalation.csv")

display(clf_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


## Experiment A: Classification

### 3.1 Problem Framing

The first experiment focuses on predicting whether a support ticket will be escalated.

This is a classification problem because the target variable, `escalated`, represents different categories rather than a continuous numerical value.

The target variable is:

- `0`: The ticket is not escalated.
- `1`: The ticket is escalated.

The `ticket_id` column is excluded because it is an identifier and does not provide meaningful information for predicting escalation.

The objective is to train and compare three classification algorithms:

1. Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Tree Classifier

The F1-score is selected as the primary evaluation metric because it considers both precision and recall. This is useful when the classes are imbalanced and identifying escalated tickets is important.

### 3.2 Understanding the Dataset

Before training the models, the dataset is inspected to understand its size, column types, missing values, and target distribution.

This step helps identify numerical and categorical features and determine whether the dataset contains missing values.

The distribution of the target variable is also examined to check whether the two classes are balanced.

In [3]:
print(clf_df.shape)

print(clf_df.info())

print(clf_df.isnull().sum())

print(clf_df["escalated"].value_counts())

(900, 7)
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               900 non-null    str    
 1   complexity_score        878 non-null    float64
 2   team_experience_years   864 non-null    float64
 3   priority                887 non-null    str    
 4   channel                 891 non-null    str    
 5   customer_tenure_months  869 non-null    float64
 6   escalated               900 non-null    int64  
dtypes: float64(3), int64(1), str(3)
memory usage: 49.3 KB
None
ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64
escalated
0    709
1    191
Name: count, dtype: int64


### 3.3 Separating Features and Target

The dataset is divided into input features (X) and the target variable (y).

- X contains the information used by the model to make predictions.
- y contains the actual outcome that the model is expected to predict.

The `ticket_id` column is removed because it is only an identifier. The `escalated` column is also excluded from X because it is the target variable.

This prevents the model from using the answer as one of its input features.


In [29]:
X = clf_df.drop(columns=["ticket_id", "escalated"])

y = clf_df["escalated"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (900, 5)
Target Shape: (900,)


### 3.4 Train-Test Split

The dataset is divided into training and testing sets using `train_test_split()`.

- 80% of the data is used for training.
- 20% of the data is reserved for testing.

The `stratify=y` parameter ensures that the training and testing sets maintain approximately the same class distribution as the original target variable.

This is particularly important for classification because the target classes may be imbalanced.

The `random_state=42` parameter ensures that the same split can be reproduced when the code is executed again.

The test set is kept separate during model training and hyperparameter tuning. It is used only for the final evaluation.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training data: (720, 5)
Testing data: (180, 5)

Training target distribution:
escalated
0    0.7875
1    0.2125
Name: proportion, dtype: float64

Testing target distribution:
escalated
0    0.788889
1    0.211111
Name: proportion, dtype: float64


### 3.5 Data Preprocessing

Machine learning models require data to be prepared before training. The preprocessing stage handles missing values, numerical features, and categorical features.

First, the numerical and categorical columns are identified using their data types.

**Numerical features:**

- Missing values are replaced using the median.
- StandardScaler is used to standardize the numerical values.

The median is used because it is less affected by extreme values than the mean. Standardization helps numerical features have a comparable scale, which is especially useful for Logistic Regression and KNN.

**Categorical features:**

- Missing values are replaced using the most frequent value.
- OneHotEncoder converts categorical values into numerical representations.

The `handle_unknown="ignore"` parameter allows the model to handle categories that were not present during training.

ColumnTransformer combines the numerical and categorical preprocessing steps.

All preprocessing is placed inside a Pipeline. This prevents data leakage because preprocessing is fitted only on the training portion of each cross-validation fold.


In [6]:
num_cols = X_train.select_dtypes(
    include=np.number
).columns

cat_cols = X_train.select_dtypes(
    exclude=np.number
).columns

In [10]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

### 3.6 Model Comparison Using Cross-Validation

Three classification algorithms are compared:

**1. Logistic Regression**

Logistic Regression is a classification algorithm that estimates the probability of an observation belonging to a particular class. It is useful as a simple and interpretable baseline model.

**2. K-Nearest Neighbors (KNN)**

KNN predicts a class based on the classes of nearby observations. It uses similarity between data points to make predictions.

**3. Decision Tree Classifier**

A Decision Tree makes predictions by splitting the data into smaller groups based on feature values. It can learn non-linear relationships and is relatively easy to understand.

To compare these models fairly, the same 5-fold StratifiedKFold cross-validation strategy is used.

The training data is divided into five folds. Each fold is used once for validation while the remaining four folds are used for training.

The F1-score is used as the primary metric. Both the mean and standard deviation are reported.

- Mean F1-score shows the average model performance across the folds.
- Standard deviation shows how much the model performance changes between folds.

A higher mean F1-score indicates better performance, while a lower standard deviation suggests more consistent performance.

In [11]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
results = []

for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )

    results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

classification_results = pd.DataFrame(results)

display(classification_results)

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,KNN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


The cross-validation results are used to compare the performance and consistency of the three classification models.

The model with the highest mean F1-score demonstrates the strongest average performance across the five folds.

The standard deviation helps identify whether a model performs consistently or whether its performance changes considerably depending on the training and validation data.

These results provide a basis for selecting a candidate model for hyperparameter tuning.

### 3.7 Hyperparameter Tuning Using GridSearchCV

Hyperparameter tuning is performed to find a suitable configuration for the selected classification model.

Logistic Regression is used for tuning. Its `C` parameter controls the strength of regularization.

- A smaller C value means stronger regularization.
- A larger C value means weaker regularization.

The parameter grid contains five possible values: 0.01, 0.1, 1, 10, and 100.

GridSearchCV evaluates the different parameter combinations using the same 5-fold cross-validation strategy.

The F1-score is used to identify the best configuration.

The preprocessing steps are included inside the pipeline so that they are fitted correctly within each cross-validation fold.

In [14]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

print("Best CV F1:", grid.best_score_)

Best Parameters: {'model__C': 10}
Best CV F1: 0.4225932078257659


The GridSearchCV results identify the best hyperparameter configuration based on the average F1-score across the validation folds.

The best estimator is automatically refitted using the complete training dataset.

The selected pipeline is then used for final evaluation on the test set.

### 3.8 Final Model Evaluation

After hyperparameter tuning, the selected model is evaluated using the test dataset.

The test data has not been used during model training or hyperparameter selection. Therefore, it provides an estimate of how well the model performs on unseen data.

The classification report provides:

- **Precision:** The proportion of predicted positive cases that are actually positive.
- **Recall:** The proportion of actual positive cases correctly identified.
- **F1-score:** The balance between precision and recall.
- **Support:** The number of actual observations in each class.

A confusion matrix is also used to examine correct and incorrect predictions.

For this experiment, false negatives are particularly important because they represent escalated tickets that the model incorrectly predicts as not escalated.

In [15]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180

[[137   5]
 [ 30   8]]


### Interpretation of Classification Results

The best hyperparameter found by GridSearchCV was:

- C = 10

The best cross-validation F1-score was approximately 0.4226.

On the test dataset, the model achieved an accuracy of 0.81.

For class 0 (not escalated), the model achieved:
- Precision: 0.82
- Recall: 0.96
- F1-score: 0.89

For class 1 (escalated), the model achieved:
- Precision: 0.62
- Recall: 0.21
- F1-score: 0.31

The confusion matrix was:

| Actual / Predicted | Not Escalated (0) | Escalated (1) |
|---|---:|---:|
| Not Escalated (0) | 137 | 5 |
| Escalated (1) | 30 | 8 |

The model correctly identified 137 non-escalated tickets and 8 escalated tickets.

However, it incorrectly classified 30 escalated tickets as not escalated. These are false negatives. It also incorrectly classified 5 non-escalated tickets as escalated, which are false positives.

Although the model achieved 81% accuracy, its recall for escalated tickets was only 0.21. This means that it identified only a small proportion of the actual escalated tickets.

Therefore, accuracy alone does not provide a complete understanding of the model's performance. The low recall for class 1 indicates that the model needs improvement in identifying escalated tickets.

Possible improvements include testing class weighting, adjusting the classification threshold, and comparing other models or configurations.

## Experiment B: Regression

### 4.1 Problem Framing

The second experiment focuses on predicting the resolution time of support tickets in hours.

This is a regression problem because the target variable, `resolution_time_hours`, represents a continuous numerical value.

The objective is to predict how many hours a support ticket will take to be resolved.

The `ticket_id` column is excluded because it is an identifier rather than a meaningful predictive feature.

Three regression algorithms are compared:

1. Linear Regression
2. Ridge Regression
3. Decision Tree Regressor

Mean Absolute Error (MAE) is selected as the primary evaluation metric because it measures the average absolute difference between actual and predicted values.

A lower MAE indicates that the model's predictions are closer to the actual resolution times.

In [19]:
reg_df = pd.read_csv("resolution_time_regression.csv")

display(reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


### 4.2 Understanding the Dataset

The regression dataset is inspected to understand its structure, data types, missing values, and target variable.

This helps identify the features that will be used for prediction and determine the preprocessing steps required before training the regression models.

In [30]:
print("Dataset Shape:", reg_df.shape)

print("\nDataset Information:")
reg_df.info()

print("\nMissing Values:")
display(reg_df.isnull().sum())

print("\nResolution Time Statistics:")
display(reg_df["resolution_time_hours"].describe())

Dataset Shape: (800, 7)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               800 non-null    str    
 1   complexity_score        784 non-null    float64
 2   team_experience_years   776 non-null    float64
 3   priority                800 non-null    str    
 4   channel                 800 non-null    str    
 5   customer_tenure_months  780 non-null    float64
 6   resolution_time_hours   800 non-null    float64
dtypes: float64(4), str(3)
memory usage: 43.9 KB

Missing Values:


ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64


Resolution Time Statistics:


count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64

The descriptive statistics provide information about the distribution of resolution times, including the minimum, maximum, mean, and standard deviation.

The maximum and mean values can help identify whether the dataset contains unusually large resolution times.

These unusually large values may represent outliers, which can influence the performance of regression models.

### 4.3 Separating Features and Target

The regression dataset is separated into input features (X) and the target variable (y).

The `resolution_time_hours` column is used as the target because it is the value we want to predict.

The `ticket_id` column is removed because it is only an identifier and should not be used as a predictive feature.

In [31]:
X_reg = reg_df.drop(
    columns=["ticket_id", "resolution_time_hours"]
)

y_reg = reg_df["resolution_time_hours"]

print("Features Shape:", X_reg.shape)
print("Target Shape:", y_reg.shape)

Features Shape: (800, 5)
Target Shape: (800,)


### 4.4 Train-Test Split

The regression dataset is divided into training and testing sets.

80% of the data is used for training, while the remaining 20% is reserved for final testing.

The `random_state=42` parameter ensures that the same split can be reproduced.

Unlike the classification experiment, stratification is not used because the target variable is continuous rather than a set of classes.

The test set remains separate during model comparison and hyperparameter tuning.

In [33]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)
print("Training Set:", X_train_reg.shape)
print("Testing Set:", X_test_reg.shape)

Training Set: (640, 5)
Testing Set: (160, 5)


### 4.5 Data Preprocessing

The regression dataset is preprocessed using the same general approach as the classification experiment.

Numerical features:
- Missing values are replaced using the median.
- StandardScaler standardizes numerical values.

Categorical features:
- Missing values are replaced using the most frequent value.
- OneHotEncoder converts categorical values into numerical representations.

ColumnTransformer combines both preprocessing operations.

The preprocessing steps are included inside a Pipeline to prevent data leakage during cross-validation.

This ensures that the preprocessing transformations are learned only from the training portion of each fold.

In [23]:
num_cols_reg = X_train_reg.select_dtypes(
    include=np.number
).columns

cat_cols_reg = X_train_reg.select_dtypes(
    exclude=np.number
).columns

num_pipe_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer([
    ("num", num_pipe_reg, num_cols_reg),
    ("cat", cat_pipe_reg, cat_cols_reg)
])

### 4.6 Model Comparison Using Cross-Validation

Three regression algorithms are compared using 5-fold cross-validation.

**1. Linear Regression**

Linear Regression predicts a continuous target by finding a linear relationship between the input features and the target variable.

**2. Ridge Regression**

Ridge Regression is a regularized version of Linear Regression. It adds a penalty to large coefficients to help reduce overfitting.

**3. Decision Tree Regressor**

A Decision Tree Regressor makes predictions by dividing the data into smaller groups based on feature values. It can capture non-linear relationships between features and the target.

KFold cross-validation is used with five folds. Each fold is used once for validation while the remaining folds are used for training.

Mean Absolute Error (MAE) is used to compare the models.

Since Scikit-learn returns negative values for the `neg_mean_absolute_error` scoring method, the scores are multiplied by -1 to obtain positive MAE values.

A lower mean MAE indicates better average prediction performance. The standard deviation shows how much the performance varies between folds.

In [24]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}

In [25]:
from sklearn.model_selection import KFold

reg_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [26]:
reg_results = []

for name, model in reg_models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring="neg_mean_absolute_error"
    )

    mae_scores = -scores["test_score"]

    reg_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

regression_results = pd.DataFrame(reg_results)

display(regression_results)

,Model,Mean MAE,Std MAE
0,Linear Regression,2.035134,0.171650
1,Ridge Regression,2.035127,0.171404
2,Decision Tree,3.239097,0.053807


The cross-validation results show how the three regression models perform across five validation folds.

The model with the lowest mean MAE has the smallest average prediction error.

The standard deviation provides information about the consistency of each model across the folds.

A model with a low mean MAE and relatively low standard deviation may be a suitable candidate for final selection.

The results are used to guide the selection of a model for hyperparameter tuning.

### 4.7 Hyperparameter Tuning Using GridSearchCV

Ridge Regression is selected for hyperparameter tuning.

The main hyperparameter considered is `alpha`, which controls the strength of regularization.

- A smaller alpha applies a weaker penalty.
- A larger alpha applies a stronger penalty.

The parameter grid contains different alpha values. GridSearchCV evaluates these values using the same 5-fold cross-validation strategy.

The negative MAE scoring method is used because the objective is to minimize prediction error.

The best configuration is selected based on the lowest average MAE.

In [27]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

ridge_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_search = GridSearchCV(
    ridge_pipeline,
    ridge_grid,
    cv=reg_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

ridge_search.fit(X_train_reg, y_train_reg)

print("Best Parameters:", ridge_search.best_params_)

print("Best CV MAE:", -ridge_search.best_score_)

Best Parameters: {'model__alpha': 0.1}
Best CV MAE: 2.035118085555478


GridSearchCV evaluates the selected alpha values and identifies the configuration with the best cross-validation performance.

The negative best score is converted into a positive MAE value for easier interpretation.

The selected Ridge Regression pipeline is then used for final testing.

### 4.8 Final Regression Model Evaluation

The selected regression model is evaluated using the test dataset.

Mean Absolute Error (MAE) is calculated to measure the average absolute difference between the predicted and actual resolution times.

Root Mean Squared Error (RMSE) is also calculated to understand the effect of larger prediction errors.

MAE treats errors proportionally, while RMSE gives more weight to larger errors because the errors are squared before calculating the average.

The test dataset is used only for this final evaluation.

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

best_reg_model = ridge_search.best_estimator_

y_pred_reg = best_reg_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)

rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print("Test MAE:", mae)

print("Test RMSE:", rmse)

Test MAE: 2.2300861441806563
Test RMSE: 3.4027561084563143


### Interpretation of Regression Results

The final regression model is evaluated using the test dataset.

The MAE represents the average absolute prediction error in hours. For example, a lower MAE means that the predicted resolution times are closer to the actual resolution times on average.

The RMSE measures prediction errors differently because it gives greater weight to larger errors.

If RMSE is considerably higher than MAE, this may indicate that some predictions have relatively large errors.

Outliers can affect regression models, particularly models that are sensitive to large numerical values.

MAE and RMSE may prefer different models because they measure prediction errors differently. MAE treats all absolute errors proportionally, while RMSE penalizes larger errors more strongly.

Therefore, the choice of evaluation metric should depend on the problem and the importance of avoiding large prediction errors.

### 4.9 Limitations of the Regression Model

The regression models have some limitations.

First, the quality of predictions depends on the features available in the dataset. Missing or insufficient information may reduce prediction accuracy.

Second, outliers in resolution time can affect model performance, especially when using metrics such as RMSE.

Third, the selected model may perform differently on new data that has a different distribution from the training dataset.

Future improvements could include investigating outliers, testing additional hyperparameter values, and evaluating other regression algorithms.

# 5. Overall Conclusion

This lab applied machine learning techniques to two different problems: classification and regression.

In Experiment A, classification models were used to predict whether support tickets would be escalated. Logistic Regression was tuned using GridSearchCV. The final test results showed that the model performed well at identifying non-escalated tickets but had difficulty identifying escalated tickets. This demonstrates why accuracy alone is not sufficient when evaluating a model with imbalanced classes.

In Experiment B, regression models were used to predict support ticket resolution time. Linear Regression, Ridge Regression, and Decision Tree Regression were compared using 5-fold cross-validation and MAE. Ridge Regression was tuned using GridSearchCV, and the selected model was evaluated on the test dataset.

Both experiments used preprocessing pipelines to handle missing values, scale numerical features, and encode categorical features. Keeping preprocessing inside the pipelines helped prevent data leakage during cross-validation.

Overall, the lab demonstrated the importance of selecting appropriate evaluation metrics, comparing models using cross-validation, tuning hyperparameters, and evaluating final models on unseen test data.

# 6. Reflection Questions

### 1. Why stratify the classification split?

Stratification ensures that the training and testing sets maintain approximately the same class distribution as the original dataset. This is important when the target classes are imbalanced because it helps both sets represent the original data more accurately.

### 2. Why must preprocessing be inside the pipeline?

Preprocessing must be inside the pipeline to prevent data leakage. During cross-validation, the preprocessing steps are fitted only on the training portion of each fold. This prevents information from the validation data from influencing the preprocessing process.

### 3. What does fold variability tell you?

Fold variability shows how much a model's performance changes across different cross-validation folds. A low standard deviation indicates more consistent performance, while a high standard deviation suggests that the model may be sensitive to the particular data used for training and validation.

### 4. Why should the test set be evaluated only once?

The test set should be evaluated only once because it is intended to provide an unbiased estimate of how the final selected model performs on unseen data. Repeatedly checking test results and making model changes based on them can lead to overfitting to the test set.

### 5. What deployment constraints might change your choice?

Deployment constraints such as prediction speed, memory requirements, computational resources, interpretability, maintenance, and the cost of incorrect predictions may influence the final model choice. For example, a slightly less accurate model may be preferred if it is faster, easier to maintain, or more interpretable.